# init_db.ipynb

In [1]:
%%configure
{
    "--datalake-formats": "iceberg",
    "--conf": "spark.sql.extensions=org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions --conf spark.sql.catalog.glue_catalog=org.apache.iceberg.spark.SparkCatalog --conf spark.sql.catalog.glue_catalog.catalog-impl=org.apache.iceberg.aws.glue.GlueCatalog --conf spark.sql.catalog.glue_catalog.io-impl=org.apache.iceberg.aws.s3.S3FileIO --conf spark.sql.iceberg.handle-timestamp-without-timezone=true --conf spark.sql.catalog.glue_catalog.warehouse=s3://002404305765-aws-s3-silver/athena/"
}

UsageError: Cell magic `%%configure` not found.


In [ ]:
%idle_timeout 120
%glue_version 5.0
%worker_type G.1X
%number_of_workers 2

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
  
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

In [ ]:
# Definición de variables
BUCKET_LOCATION = "s3://<BUCKET_NAME>/databases/"

In [ ]:
# Crear Databases
spark.sql(f"CREATE DATABASE IF NOT EXISTS bronze LOCATION '{BUCKET_LOCATION}'")
spark.sql(f"CREATE DATABASE IF NOT EXISTS silver LOCATION '{BUCKET_LOCATION}'")
spark.sql(f"CREATE DATABASE IF NOT EXISTS metadata LOCATION '{BUCKET_LOCATION}'")

In [ ]:
# Crear Tables
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS glue_catalog.bronze.events (
        event_date DATE,
        user_mail STRING,
        event STRING,
        system STRING
    )
    USING iceberg
    LOCATION '{BUCKET_LOCATION}/iceberg.db/bronze/events/'
    PARTITIONED BY (event_date)
    TBLPROPERTIES (
        'table_type'='iceberg',
        'format-version' = '2',
        'write_compression' = 'snappy'
    )
""")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS glue_catalog.metadata.etl_audit (
        uuid STRING,
        etl_name STRING,
        status STRING,
        start_date DATE,
        end_date DATE,
        catchup BOOLEAN,
        backfill BOOLEAN,
        updated_at TIMESTAMP
    )
    USING iceberg
    LOCATION '{BUCKET_LOCATION}/iceberg.db/metadata/etl_audit/'
        TBLPROPERTIES (
        'table_type'='iceberg',
        'format-version' = '2',
        'write_compression' = 'snappy'
    )
""")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS glue_catalog.silver.events (
        uuid STRING,
        event_date DATE,
        user_mail STRING,
        event STRING,
        system STRING
    )
    USING iceberg
    LOCATION '{BUCKET_LOCATION}/iceberg.db/silver/events/'
    PARTITIONED BY (event_date)
    TBLPROPERTIES (
        'table_type'='iceberg',
        'format-version' = '2',
        'write_compression' = 'snappy'
    )
""")

In [ ]:
# Limpia tablas en caso de volver a ejecutar el notebook y crea snapshot inicial
spark.sql("TRUNCATE TABLE glue_catalog.bronze.events")
spark.sql("TRUNCATE TABLE glue_catalog.silver.events")
spark.sql("TRUNCATE TABLE glue_catalog.metadata.etl_audit")

In [ ]:
# Cargar 30 días a partir de ayer
import random
import string
from datetime import datetime, timedelta
from pyspark.sql.types import StructType, StructField, StringType, DateType

yesterday = datetime.now() - timedelta(days=1)
actions = ['comment', 'post', 'view', 'search', 'profile']
systems = ['alpha', 'beta', 'gamma', 'delta']

records = []

for i in range(30):
    current_date = (yesterday - timedelta(days=i)).date()
    
    for _ in range(3):
        random_letters = ''.join(random.choices(string.ascii_lowercase, k=3))
        mail_usuario = f"{random_letters}@mail.com"
        accion = random.choice(actions)
        sistema = random.choice(systems)
        
        records.append((current_date, mail_usuario, accion, sistema))

schema = StructType([
    StructField("event_date", DateType(), True),
    StructField("user_mail", StringType(), True),
    StructField("event", StringType(), True),
    StructField("system", StringType(), True)
])

df_bronze = spark.createDataFrame(records, schema)

df_bronze.writeTo("glue_catalog.bronze.events").append()

In [ ]:
# Ver datos
spark.sql("SELECT * FROM glue_catalog.bronze.events").show()

In [ ]:
# Insertar registros de auditoría de ejemplo para pruebas
import uuid
from datetime import datetime, timedelta
from pyspark.sql.types import StructType, StructField, StringType, DateType, BooleanType, TimestampType

now = datetime.now()
audit_records = [
    (str(uuid.uuid4()), 'events', 'success', (now - timedelta(days=7)).date(), (now - timedelta(days=7)).date(), True, False, (now - timedelta(days=7))),
    (str(uuid.uuid4()), 'events', 'failed', (now - timedelta(days=6)).date(), (now - timedelta(days=6)).date(), True, False, (now - timedelta(days=6))),
    (str(uuid.uuid4()), 'events', 'failed', (now - timedelta(days=5)).date(), (now - timedelta(days=5)).date(), True, False, (now - timedelta(days=5)))
]

schema_audit = StructType([
    StructField("uuid", StringType(), False),
    StructField("etl_name", StringType(), True),
    StructField("status", StringType(), True),
    StructField("start_date", DateType(), True),
    StructField("end_date", DateType(), True),
    StructField("catchup", BooleanType(), True),
    StructField("backfill", BooleanType(), True),
    StructField("updated_at", TimestampType(), True)
])

df_audit = spark.createDataFrame(audit_records, schema=schema_audit)

df_audit.writeTo("glue_catalog.metadata.etl_audit").append()

In [ ]:
# Ver datos
spark.sql("SELECT * FROM glue_catalog.metadata.etl_audit").show()

In [ ]:
%stop_session